# 🏺 QwenCleo-ASR — vLLM Serving & True Streaming

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

QwenCleo inherits Qwen3-ASR's **token-by-token streaming** via vLLM.

> ⚠️ Needs a recent CUDA. On Colab pick a **GPU** runtime; vLLM's Qwen3-ASR support is on the **nightly** build, which is large — the install cell can take several minutes.

## 1. Install vLLM (nightly) + the client

In [ ]:
# vLLM nightly has day-0 Qwen3-ASR support
!pip install -q -U vllm --pre \
    --extra-index-url https://wheels.vllm.ai/nightly \
    --index-strategy unsafe-best-match || pip install -q -U vllm
!pip install -q 'qwencleo-asr[vllm-client]'

## 2. Start a vLLM server in the background

In [ ]:
import subprocess, time, os
proc = subprocess.Popen(
    ['vllm', 'serve', 'mohammedaly22/QwenCleo-ASR'])
print('starting vLLM (downloads + loads the model)...')
# wait for the server to come up
import requests
for _ in range(60):
    try:
        requests.get('http://localhost:8000/v1/models', timeout=2)
        print('server up'); break
    except Exception:
        time.sleep(10)

## 3. Sample audio

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

## 4. TRUE streaming — deltas arrive token by token

In [ ]:
from qwencleo_asr import stream_vllm

for delta in stream_vllm('sample.wav', language='Arabic'):
    print(delta, end='', flush=True)
print()

## 5. One-shot via the OpenAI-compatible API

In [ ]:
from qwencleo_asr import transcribe_vllm
print(transcribe_vllm('sample.wav'))

## 6. OpenAI SDK directly (the documented form)

In [ ]:
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')
with open('sample.wav','rb') as f:
    print(client.audio.transcriptions.create(
        model='mohammedaly22/QwenCleo-ASR', file=f.read()).text)

## 7. Stop the server

In [ ]:
proc.terminate()